In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from ast import literal_eval
import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report

PHASE0_DIR = Path("/kaggle/input/datasets/<username>/rsna-atd-phase0-preprocessed/processed")
MANIFEST_PATH = PHASE0_DIR / "manifest.csv"
LABELS_PATH = PHASE0_DIR / "subset_labels.csv"

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = 256
N_SLICES_PER_ORGAN = 32
ORGAN_NUM_CLASSES = {"liver": 3, "spleen": 3, "kidney": 3, "bowel": 2}
SEEDS = [42, 123, 2024, 7, 555]
N_FOLDS = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
manifest_df = pd.read_csv(MANIFEST_PATH)
manifest_df['output_path'] = manifest_df['output_path'].apply(lambda p: str(PHASE0_DIR / Path(p).name))

sample = manifest_df.iloc[0]
print("Rewritten path exists:", Path(sample['output_path']).exists())

# Safe parsing instead of eval()
manifest_df['organs_cropped'] = manifest_df['organs_cropped'].apply(literal_eval)
manifest_df['resize_factors'] = manifest_df['resize_factors'].apply(literal_eval)

print(f"Total series: {len(manifest_df)}, with organ crops: {(manifest_df['organs_cropped'].apply(len) > 0).sum()}")

Rewritten path exists: True
Total series: 1803, with organ crops: 206


In [3]:
labels_df = pd.read_csv(LABELS_PATH)

# Defensive checks from the code review
if labels_df['patient_id'].duplicated().any():
    dupes = labels_df.loc[labels_df['patient_id'].duplicated(keep=False), 'patient_id'].unique()
    raise ValueError(f"Duplicate patient labels found: {dupes[:10]}")

required_labels = [
    "bowel_injury",
    "liver_healthy", "liver_low", "liver_high",
    "spleen_healthy", "spleen_low", "spleen_high",
    "kidney_healthy", "kidney_low", "kidney_high",
]
manifest_labeled_check = manifest_df.merge(labels_df, on="patient_id", how="left", validate="many_to_one")
missing = manifest_labeled_check[required_labels].isna().any(axis=1)
if missing.any():
    raise ValueError(f"{missing.sum()} manifest rows have missing injury labels")

print("Label validation passed: no duplicate patients, no missing required labels.")

Label validation passed: no duplicate patients, no missing required labels.


In [4]:
strat_df = labels_df[['patient_id', 'any_injury', 'bowel_injury', 'extravasation_injury',
                       'kidney_high', 'liver_high', 'spleen_high']].copy()
strat_df['strat_key'] = (
    strat_df['bowel_injury'].astype(str) + "_" +
    strat_df['extravasation_injury'].astype(str) + "_" +
    (strat_df[['kidney_high','liver_high','spleen_high']].sum(axis=1) > 0).astype(int).astype(str)
)
bucket_counts = strat_df['strat_key'].value_counts()
rare_buckets = bucket_counts[bucket_counts < 2].index
strat_df.loc[strat_df['strat_key'].isin(rare_buckets), 'strat_key'] = "other"

train_patients, val_patients = train_test_split(
    strat_df['patient_id'], test_size=0.15, random_state=42, stratify=strat_df['strat_key']
)
train_patients, val_patients = set(train_patients), set(val_patients)
assert train_patients.isdisjoint(val_patients)

manifest_labeled = manifest_df.merge(labels_df, on='patient_id', how='inner', validate="many_to_one")
manifest_labeled['split'] = manifest_labeled['patient_id'].apply(
    lambda p: 'train' if p in train_patients else ('val' if p in val_patients else 'unused')
)
organ_manifest = manifest_labeled[manifest_labeled['organs_cropped'].apply(len) > 0].reset_index(drop=True)

print(f"Series usable for organ-branch training: {len(organ_manifest)}")
print(organ_manifest['split'].value_counts())

Series usable for organ-branch training: 206
split
train    173
val       33
Name: count, dtype: int64


In [5]:
def get_organ_label(row, organ):
    if organ == "bowel":
        return int(row['bowel_injury'])
    healthy, low, high = row[f'{organ}_healthy'], row[f'{organ}_low'], row[f'{organ}_high']
    return int(np.argmax([healthy, low, high]))

class OrganCropDataset(Dataset):
    def __init__(self, manifest_df, organ, n_slices=N_SLICES_PER_ORGAN, img_size=TARGET_SIZE):
        self.df = manifest_df[manifest_df['organs_cropped'].apply(lambda x: organ in x)].reset_index(drop=True)
        self.organ = organ
        self.n_slices = n_slices
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with np.load(row['output_path']) as data:
            vol = data[self.organ].astype(np.float32) / 255.0
        label = get_organ_label(row, self.organ)
        weight = self._get_sample_weight(row)
        vol = torch.from_numpy(vol).unsqueeze(1)
        return vol, label, weight

    def _get_sample_weight(self, row):
        if self.organ == "bowel":
            return 2.0 if row['bowel_injury'] == 1 else 1.0
        healthy, low, high = row[f'{self.organ}_healthy'], row[f'{self.organ}_low'], row[f'{self.organ}_high']
        if high == 1: return 4.0
        if low == 1: return 2.0
        return 1.0

def make_loader(manifest_df, organ, split, batch_size=4, num_workers=0):
    df_split = manifest_df[manifest_df['split'] == split]
    ds = OrganCropDataset(df_split, organ)
    return DataLoader(ds, batch_size=batch_size, shuffle=(split == 'train'),
                       num_workers=num_workers, pin_memory=True)

!pip install -q timm
import timm

class SliceAttentionPool(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(feat_dim, 128), nn.Tanh(), nn.Linear(128, 1))
    def forward(self, feats):
        weights = torch.softmax(self.attn(feats), dim=1)
        return (feats * weights).sum(dim=1)

class OrganClassifier(nn.Module):
    def __init__(self, num_classes, backbone_name="efficientnet_b0"):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0, in_chans=1)
        feat_dim = self.backbone.num_features
        self.pool = SliceAttentionPool(feat_dim)
        self.head = nn.Linear(feat_dim, num_classes)
    def forward(self, x):
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)
        feats = self.backbone(x).view(B, S, -1)
        return self.head(self.pool(feats))

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

In [6]:
class OrdinalPresentLabelUpweightingLoss(nn.Module):
    def __init__(self, class_counts, beta=0.999, ordinal_alpha=0.5, present_boost=1.5, focal_gamma=1.0):
        super().__init__()
        class_counts = np.array(class_counts, dtype=np.float32)
        if np.any(class_counts == 0):
            raise ValueError(f"Zero-count class in training split: {class_counts.tolist()}")
        effective_num = 1.0 - np.power(beta, class_counts)
        weights = (1.0 - beta) / effective_num
        weights = weights / weights.sum() * len(class_counts)
        self.class_weights = torch.tensor(weights, dtype=torch.float32)
        self.ordinal_alpha = ordinal_alpha
        self.present_boost = present_boost
        self.focal_gamma = focal_gamma
        self.num_classes = len(class_counts)

    def forward(self, logits, targets, sample_weight=None):
        device = logits.device
        class_weights = self.class_weights.to(device)
        probs = torch.softmax(logits, dim=1)
        log_probs = torch.log_softmax(logits, dim=1)
        target_probs = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_factor = (1 - target_probs) ** self.focal_gamma
        ce = -log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        cls_w = class_weights[targets]
        base_loss = focal_factor * cls_w * ce
        class_ranks = torch.arange(self.num_classes, device=device).float()
        expected_rank = (probs * class_ranks.unsqueeze(0)).sum(dim=1)
        ordinal_penalty = (expected_rank - targets.float()) ** 2
        present_mask = (targets > 0).float()
        present_weight = 1.0 + (self.present_boost - 1.0) * present_mask
        loss = present_weight * (base_loss + self.ordinal_alpha * ordinal_penalty)
        if sample_weight is not None:
            loss = loss * sample_weight
        return loss.mean()

# Explicit, documented choice: the confirmed sweep results did NOT apply the
# dataset's external severity sample_weight on top of the ordinal loss's own
# internal weighting. Kept False here to match already-reported results;
# set True only if deliberately re-running with that additional term.
ORDINAL_USE_SAMPLE_WEIGHT = False

def get_class_counts(manifest_df, organ):
    labels = manifest_df.apply(lambda row: get_organ_label(row, organ), axis=1)
    return [int((labels == c).sum()) for c in range(ORGAN_NUM_CLASSES[organ])]

In [7]:
def make_patient_folds(labels_df, organ_manifest, n_folds=N_FOLDS, seed=42):
    patients_with_crops = organ_manifest['patient_id'].unique()
    patient_labels = labels_df[labels_df['patient_id'].isin(patients_with_crops)].drop_duplicates(
        subset='patient_id').reset_index(drop=True)

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold_assignment = {}
    for fold_idx, (_, test_idx) in enumerate(skf.split(patient_labels['patient_id'], patient_labels['any_injury'])):
        for pid in patient_labels.iloc[test_idx]['patient_id']:
            fold_assignment[pid] = fold_idx

    organ_manifest_cv = organ_manifest.copy()
    organ_manifest_cv['fold'] = organ_manifest_cv['patient_id'].map(fold_assignment)
    if organ_manifest_cv['fold'].isna().any():
        raise ValueError("Some rows have no fold assignment")
    return organ_manifest_cv

organ_manifest_cv = make_patient_folds(labels_df, organ_manifest, n_folds=N_FOLDS)
print(organ_manifest_cv.groupby('fold').size())
print(organ_manifest_cv.groupby('fold')['any_injury'].mean())

fold
0    42
1    43
2    43
3    38
4    40
dtype: int64
fold
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: any_injury, dtype: float64


In [8]:
def make_cv_loader(manifest_df, organ, fold, mode, batch_size=4):
    df_split = manifest_df[manifest_df['fold'] != fold] if mode == 'train' else manifest_df[manifest_df['fold'] == fold]
    ds = OrganCropDataset(df_split, organ)
    return DataLoader(ds, batch_size=batch_size, shuffle=(mode == 'train'), num_workers=0, pin_memory=True)

def train_organ_cv(organ, manifest_cv, method="baseline", epochs=5, seed=42,
                   ordinal_alpha=0.5, present_boost=1.5, focal_gamma=1.0, n_folds=N_FOLDS):
    fold_results = []
    
    for fold in range(n_folds):
        set_seed(seed)
        
        # NOTE: Be sure to pass num_workers=2 (or more) in make_cv_loader for speed
        train_loader = make_cv_loader(manifest_cv, organ, fold, 'train', batch_size=4)
        val_loader = make_cv_loader(manifest_cv, organ, fold, 'val', batch_size=4)
        
        if len(val_loader.dataset) == 0 or len(train_loader.dataset) == 0:
            print(f"[{organ}] fold {fold}: empty split, skipping")
            continue

        model = OrganClassifier(num_classes=ORGAN_NUM_CLASSES[organ]).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-4)
        
        # 1. Initialize the AMP GradScaler
        scaler = torch.amp.GradScaler('cuda')

        if method == "baseline":
            criterion = nn.CrossEntropyLoss(reduction='none')
        else:
            try:
                class_counts = get_class_counts(train_loader.dataset.df, organ)
            except ValueError as e:
                print(f"[{organ}] fold {fold}: {e}, skipping")
                continue
            criterion = OrdinalPresentLabelUpweightingLoss(
                class_counts, ordinal_alpha=ordinal_alpha, present_boost=present_boost, focal_gamma=focal_gamma)

        best_val_loss = float('inf')
        best_preds = None
        
        print(f"\n--- Training {organ.upper()} | Method: {method} | Fold {fold} ---")
        
        for epoch in range(epochs):
            model.train()
            train_losses = []
            
            for vol, label, weight in train_loader:
                vol, label, weight = vol.to(device), label.to(device), weight.to(device).float()
                optimizer.zero_grad()
                
                # 2. Autocast forward pass for Mixed Precision
                with torch.amp.autocast('cuda'):
                    logits = model(vol)
                    if method == "baseline":
                        loss = (criterion(logits, label) * weight).mean()
                    else:
                        sw = weight if ORDINAL_USE_SAMPLE_WEIGHT else None
                        loss = criterion(logits, label, sample_weight=sw)
                
                # 3. Scaled backward pass and optimizer step
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                train_losses.append(loss.item())

            # Validation Loop
            model.eval()
            val_losses, val_preds, val_labels = [], [], []
            
            with torch.no_grad():
                for vol, label, weight in val_loader:
                    vol, label, weight = vol.to(device), label.to(device), weight.to(device).float()
                    
                    # 4. Use Autocast during validation for speed
                    with torch.amp.autocast('cuda'):
                        logits = model(vol)
                        if method == "baseline":
                            loss = (criterion(logits, label) * weight).mean()
                        else:
                            sw = weight if ORDINAL_USE_SAMPLE_WEIGHT else None
                            loss = criterion(logits, label, sample_weight=sw)
                            
                    val_losses.append(loss.item())
                    val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
                    val_labels.extend(label.cpu().numpy().tolist())

            train_loss_epoch = np.mean(train_losses)
            val_loss_epoch = np.mean(val_losses)
            val_acc_epoch = np.mean(np.array(val_preds) == np.array(val_labels))
            
            print(f"  Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss_epoch:.4f} | Val Loss: {val_loss_epoch:.4f} | Val Acc: {val_acc_epoch:.4f}")

            # 5. Save the actual model weights when validation loss improves
            if val_loss_epoch < best_val_loss:
                best_val_loss = val_loss_epoch
                best_preds = (val_labels, val_preds)
                
                model_save_path = OUTPUT_DIR / f"{organ}_{method}_fold{fold}_best.pth"
                torch.save(model.state_dict(), model_save_path)
                print(f"  [>] Saved new best model to: {model_save_path.name}")

        # Post-training evaluation for the fold
        val_labels, val_preds = best_preds
        val_acc_final = np.mean(np.array(val_preds) == np.array(val_labels))
        report = classification_report(val_labels, val_preds, labels=list(range(ORGAN_NUM_CLASSES[organ])),
                                       output_dict=True, zero_division=0)
        
        fold_results.append({
            "organ": organ, 
            "method": method, 
            "fold": fold,
            "val_loss": best_val_loss, 
            "val_acc": val_acc_final,
            "per_class_recall": json.dumps({str(k): v['recall'] for k, v in report.items() if k.isdigit()}),
            "n_val": len(val_labels),
        })
        
    return pd.DataFrame(fold_results)

In [9]:
CV_CANDIDATES = {
    "liver":  [{"method": "baseline"}, {"method": "ordinal", "ordinal_alpha": 0.5, "present_boost": 1.5}],
    "spleen": [{"method": "baseline"}, {"method": "ordinal", "ordinal_alpha": 1.0, "present_boost": 1.0},
               {"method": "ordinal", "ordinal_alpha": 3.0, "present_boost": 1.0}],
    "kidney": [{"method": "baseline"}, {"method": "ordinal", "ordinal_alpha": 1.0, "present_boost": 1.0}],
}

cv_results_path = OUTPUT_DIR / "cv_results.csv"
cv_all_df = pd.read_csv(cv_results_path) if cv_results_path.exists() else pd.DataFrame()
done_keys = set(zip(cv_all_df.get('organ', []), cv_all_df.get('method', []),
                     cv_all_df.get('ordinal_alpha', []), cv_all_df.get('present_boost', [])))
all_cv_frames = [cv_all_df] if len(cv_all_df) else []

for organ, candidates in CV_CANDIDATES.items():
    for cfg in candidates:
        key = (organ, cfg["method"], cfg.get("ordinal_alpha", np.nan), cfg.get("present_boost", np.nan))
        if key in done_keys:
            print(f"Skipping {key}, already done"); continue
        print(f"\n=== CV: {organ}, {cfg} ===")
        df = train_organ_cv(organ, organ_manifest_cv, **cfg)
        df['ordinal_alpha'] = cfg.get("ordinal_alpha", np.nan)
        df['present_boost'] = cfg.get("present_boost", np.nan)
        all_cv_frames.append(df)
        pd.concat(all_cv_frames, ignore_index=True).to_csv(cv_results_path, index=False)

cv_all_df = pd.concat(all_cv_frames, ignore_index=True)


=== CV: liver, {'method': 'baseline'} ===


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]


--- Training LIVER | Method: baseline | Fold 0 ---
  Epoch 01/5 | Train Loss: 2.0805 | Val Loss: 1.8967 | Val Acc: 0.4762
  [>] Saved new best model to: liver_baseline_fold0_best.pth
  Epoch 02/5 | Train Loss: 1.7789 | Val Loss: 1.8651 | Val Acc: 0.4762
  [>] Saved new best model to: liver_baseline_fold0_best.pth
  Epoch 03/5 | Train Loss: 1.4252 | Val Loss: 1.9713 | Val Acc: 0.3810
  Epoch 04/5 | Train Loss: 0.8594 | Val Loss: 2.3394 | Val Acc: 0.3333
  Epoch 05/5 | Train Loss: 0.5465 | Val Loss: 3.2218 | Val Acc: 0.4524

--- Training LIVER | Method: baseline | Fold 1 ---
  Epoch 01/5 | Train Loss: 1.9159 | Val Loss: 2.3921 | Val Acc: 0.3721
  [>] Saved new best model to: liver_baseline_fold1_best.pth
  Epoch 02/5 | Train Loss: 1.5824 | Val Loss: 2.5974 | Val Acc: 0.3488
  Epoch 03/5 | Train Loss: 1.1659 | Val Loss: 2.9605 | Val Acc: 0.3721
  Epoch 04/5 | Train Loss: 0.5440 | Val Loss: 4.5299 | Val Acc: 0.3953
  Epoch 05/5 | Train Loss: 0.2497 | Val Loss: 4.6195 | Val Acc: 0.4884

--

In [10]:
summary = cv_all_df.groupby(['organ', 'method', 'ordinal_alpha', 'present_boost']).agg(
    mean_val_acc=('val_acc', 'mean'), std_val_acc=('val_acc', 'std'),
    mean_val_loss=('val_loss', 'mean'), n_folds=('fold', 'count')
).reset_index()
print(summary.to_string(index=False))

 organ  method  ordinal_alpha  present_boost  mean_val_acc  std_val_acc  mean_val_loss  n_folds
kidney ordinal            1.0            1.0      0.565368     0.070055       1.032763        5
 liver ordinal            0.5            1.5      0.426644     0.055318       1.075354        5
spleen ordinal            1.0            1.0      0.492491     0.103377       1.309154        5
spleen ordinal            3.0            1.0      0.538084     0.051174       2.531230        5
